# Phase 14B: Controlled Shadow Observation, Production Evidence Accumulation & Governance Readiness Audit

**Project:** Student Mental Health / Student Wellbeing Score Prediction  
**Repository:** `https://github.com/MSIVAPAPARAO13/student-wellbeing-score-prediction` (`main` branch)  
**Authoritative Date:** October 7, 2026  
**Objective:** Audit the real-world observation layer, verify dynamic shadow tracking, test historical data firewall, evaluate readiness metrics, and confirm strictly blocked promotion under **OPTION C — INSUFFICIENT EVIDENCE**.


In [1]:
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# Set project root robustly
cur = Path.cwd().resolve()
while cur.name and not (cur / "app").exists():
    if cur.parent == cur:
        break
    cur = cur.parent
PROJECT_ROOT = cur
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.governance import (
    registry_manager,
    shadow_manager,
    feedback_engine,
    GovernanceEvaluator,
    ProductionObservationRecord,
    ModelRegistryManager,
    SCORE_MIN_DOMAIN,
    SCORE_MAX_DOMAIN
)
from app.monitoring import metrics_collector
print(f"Project Root: {PROJECT_ROOT}")


Project Root: C:\Users\msiva\Music\Mental-Health-Score


## 1. Load Current Governance State & Policy
Verify that registry policies strictly forbid automatic retraining and automatic promotion, and require explicit human governance committee sign-off.


In [2]:
champ = registry_manager.get_champion()
challengers = registry_manager.get_challengers()
policy = registry_manager._data.get("governance_policy", {})

print(f"Champion Model Version: {champ.get('model_version')}")
print(f"Champion Approval Status: {champ.get('approval_status')}")
print(f"Registered Challengers: {[c.get('model_version') for c in challengers]}")
print(f"Automatic Retraining Allowed: {policy.get('automatic_retraining_allowed')}")
print(f"Automatic Promotion Allowed: {policy.get('automatic_promotion_allowed')}")
print(f"Promotion Requires Human Approval: {policy.get('promotion_requires_human_approval')}")

assert policy.get("automatic_retraining_allowed") is False, "Violation: Automatic retraining must be disabled"
assert policy.get("automatic_promotion_allowed") is False, "Violation: Automatic promotion must be disabled"
assert policy.get("promotion_requires_human_approval") is True, "Violation: Human approval must be mandatory"


Champion Model Version: phase5_tuned_extra_trees
Champion Approval Status: APPROVED_FOR_PRODUCTION
Registered Challengers: ['candidate_v1_1_fast_trees', 'candidate_v1_2_revalidated']
Automatic Retraining Allowed: False
Automatic Promotion Allowed: False
Promotion Requires Human Approval: True


## 2. Verify Cryptographic Integrity of Artifacts
Verify that production Champion, Candidate Challenger, and conformal calibration artifacts remain bit-for-bit invariant.


In [3]:
CHAMPION_EXPECTED_HASH = "a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8"
CANDIDATE_EXPECTED_HASH = "aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc"
CHAMPION_CALIB_EXPECTED_HASH = "22a8f3fc5b2863855e1c2cc0dc4f1df056d0250057d1094c93f5f66d95108d0b"
CANDIDATE_CALIB_EXPECTED_HASH = "b99c8acd2d8fc2aea604228abf1e84365e601021407409ca154ba9b8c29d60af"

def compute_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest().lower()

champ_path = PROJECT_ROOT / "models" / "phase5_tuned_extra_trees.joblib"
cand_path = PROJECT_ROOT / "models" / "candidate_v1_2_revalidated.joblib"
champ_calib_path = PROJECT_ROOT / "models" / "phase7_1_conformal_calibration.json"
cand_calib_path = PROJECT_ROOT / "models" / "candidate_v1_2_conformal_calibration.json"

h_champ = compute_sha256(champ_path)
h_cand = compute_sha256(cand_path)
h_champ_calib = compute_sha256(champ_calib_path)
h_cand_calib = compute_sha256(cand_calib_path)

print(f"Champion Model Hash:       {h_champ} (Matches: {h_champ == CHAMPION_EXPECTED_HASH})")
print(f"Candidate Model Hash:      {h_cand} (Matches: {h_cand == CANDIDATE_EXPECTED_HASH})")
print(f"Champion Calibration Hash: {h_champ_calib} (Matches: {h_champ_calib == CHAMPION_CALIB_EXPECTED_HASH})")
print(f"Candidate Calibration Hash:{h_cand_calib} (Matches: {h_cand_calib == CANDIDATE_CALIB_EXPECTED_HASH})")

assert h_champ == CHAMPION_EXPECTED_HASH
assert h_cand == CANDIDATE_EXPECTED_HASH
assert h_champ_calib == CHAMPION_CALIB_EXPECTED_HASH
assert h_cand_calib == CANDIDATE_CALIB_EXPECTED_HASH
print("ALL 4 CRYPTOGRAPHIC HASHES VERIFIED BIT-FOR-BIT.")


Champion Model Hash:       a012e7a1c0ca5c9fccb21d1e46bf3b4b4a72635204c13efdc4f93d6c636747f8 (Matches: True)
Candidate Model Hash:      aad2f208a298289de57a0a8fd4aef941be0edaf940439dca98eaaf718439cdbc (Matches: True)
Champion Calibration Hash: 22a8f3fc5b2863855e1c2cc0dc4f1df056d0250057d1094c93f5f66d95108d0b (Matches: True)
Candidate Calibration Hash:b99c8acd2d8fc2aea604228abf1e84365e601021407409ca154ba9b8c29d60af (Matches: True)
ALL 4 CRYPTOGRAPHIC HASHES VERIFIED BIT-FOR-BIT.


## 3. Inspect Shadow Observation Telemetry & Isolation Safeguards
Examine shadow serving isolation: candidate failures/timeouts never impact user responses; latency SLA adheres to warm P95 < 150 ms.


In [4]:
shadow_stat = shadow_manager.get_shadow_status()
print(f"Shadow Start Timestamp:     {shadow_stat['shadow_start_timestamp']}")
print(f"Shadow Observation State:   {shadow_stat['shadow_status']}")
print(f"Candidate Benchmark P95:    {shadow_stat['candidate_benchmark_p95_ms']} ms (SLA: < {shadow_stat['latency_sla_p95_ms']} ms)")
print(f"Live Production Latency:    {shadow_stat['latency_live_status']}")
print(f"Shadow Exceptions Caught:   {shadow_stat['shadow_exceptions']}")
print(f"Shadow Timeouts Caught:     {shadow_stat['shadow_timeouts']}")

assert shadow_stat['candidate_benchmark_p95_ms'] < shadow_stat['latency_sla_p95_ms']
assert shadow_stat['latency_sla_p95_ms'] == 150.0


Shadow Start Timestamp:     2026-10-06T09:30:00Z
Shadow Observation State:   ACTIVE
Candidate Benchmark P95:    77.58 ms (SLA: < 150.0 ms)
Live Production Latency:    DATA_NOT_AVAILABLE
Shadow Exceptions Caught:   0
Shadow Timeouts Caught:     0


## 4. Inspect Verified-Label Records & Provenance Audit
Audit the verified label counter against the mandatory 100-label governance gate.


In [5]:
label_stat = feedback_engine.get_verified_label_counter()
print(f"Verified Labels Ingested:   {label_stat['verified_count']} / {label_stat['target_count']}")
print(f"Remaining Labels Required:  {label_stat['remaining_labels']}")
print(f"Paired Production Rows:     {label_stat['paired_rows']}")
print(f"100-Label Gate Met:         {label_stat['threshold_met']}")
print(f"Data Mode:                  {label_stat['data_mode']}")

assert label_stat['verified_count'] == 0
assert label_stat['target_count'] == 100
assert label_stat['threshold_met'] is False
assert label_stat['data_mode'] == "DATA_NOT_AVAILABLE"


Verified Labels Ingested:   0 / 100
Remaining Labels Required:  100
Paired Production Rows:     0
100-Label Gate Met:         False
Data Mode:                  DATA_NOT_AVAILABLE


## 5. Enforce Historical Data Firewall
Verify that historical offline data (4,998 records) cannot contaminate post-deployment verification.


In [6]:
# Test Firewall against explicit historical tag
is_hist, reason = feedback_engine.is_historical_contamination({"is_historical": True})
print(f"Explicit historical flag rejected: {is_hist} ({reason})")
assert is_hist is True

# Test Firewall against historical prediction IDs
for pid in ["hist_001", "offline_record_42", "historical_sample_99"]:
    is_hist, reason = feedback_engine.is_historical_contamination({"prediction_id": pid})
    assert is_hist is True, f"Failed to reject historical ID: {pid}"
print("Historical ID prefixes successfully rejected.")

# Test single label ingestion rejection of historical data
result = feedback_engine.ingest_single_verified_label(
    observation_id="hist_123",
    observed_score=6.5,
    provenance={"source": "offline_training_set"}
)
print(f"Firewall rejection response: {result}")
assert result["status"] == "REJECTED"
assert "HISTORICAL_DATA_FIREWALL" in result["reason"]


Explicit historical flag rejected: True (Flagged explicitly as historical data)
Historical ID prefixes successfully rejected.
Firewall rejection response: {'status': 'REJECTED', 'observation_id': 'hist_123', 'reason': "HISTORICAL_DATA_FIREWALL: Prediction ID 'hist_123' identified as historical/offline"}


## 6. Calculate Current Observation Counts & Paired Rows
Verify that observation counters reflect cold-start shadow observation without fabrication.


In [7]:
obs_count = len(shadow_manager.get_observations())
paired_count = shadow_manager.get_paired_observation_count()
verified_count = feedback_engine.get_verified_count()

print(f"Observation Records in Memory: {obs_count}")
print(f"Paired Production Observations: {paired_count}")
print(f"Verified Post-Deployment Labels: {verified_count}")

assert verified_count == 0
assert paired_count == 0


Observation Records in Memory: 0
Paired Production Observations: 0
Verified Post-Deployment Labels: 0


## 7. Calculate Dynamic Elapsed Shadow Duration
Dynamically calculate elapsed days from formal shadow launch timestamp (`2026-10-06T09:30:00Z`).


In [8]:
shadow_stat = shadow_manager.get_shadow_status()
elapsed_days = shadow_stat['elapsed_days']
days_completed = shadow_stat['days_completed']
days_remaining = shadow_stat['days_remaining']

print(f"Shadow Start:          {shadow_manager.shadow_start_timestamp}")
print(f"Runtime Timestamp:     {shadow_stat['current_time']}")
print(f"Elapsed Duration:      {elapsed_days:.4f} days (~{elapsed_days*24:.1f} hours)")
print(f"Days Completed:        {days_completed} / 14")
print(f"Days Remaining:        {days_remaining}")
print(f"Shadow Status:         {shadow_stat['shadow_status']}")
print(f"14-Day Gate Completed: {shadow_stat['is_14_days_completed']}")

assert shadow_stat['is_14_days_completed'] is False
assert shadow_stat['shadow_status'] == "ACTIVE"
assert elapsed_days < 14.0
assert elapsed_days >= 1.0


Shadow Start:          2026-10-06T09:30:00Z
Runtime Timestamp:     2026-10-07T11:09:50.457098+00:00
Elapsed Duration:      1.0690 days (~25.7 hours)
Days Completed:        1 / 14
Days Remaining:        13
Shadow Status:         ACTIVE
14-Day Gate Completed: False


## 8. Point Prediction Metrics Readiness
Evaluate point prediction metrics on verified labels. When zero post-deployment labels exist, all metrics must report `DATA_NOT_AVAILABLE`.


In [9]:
gov_metrics = GovernanceEvaluator.evaluate_production_governance_metrics([])
print("Production Governance Metrics Audit (Empty / Cold-Start):")
for k, v in gov_metrics.items():
    print(f"  {k:30s}: {v}")

assert gov_metrics["evidence_status"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["mae"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["rmse"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["r2"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["median_absolute_error"] == "DATA_NOT_AVAILABLE"


Production Governance Metrics Audit (Empty / Cold-Start):
  evidence_status               : DATA_NOT_AVAILABLE
  sample_count                  : 0
  mae                           : DATA_NOT_AVAILABLE
  rmse                          : DATA_NOT_AVAILABLE
  r2                            : DATA_NOT_AVAILABLE
  median_absolute_error         : DATA_NOT_AVAILABLE
  max_absolute_error            : DATA_NOT_AVAILABLE
  mean_error                    : DATA_NOT_AVAILABLE
  paired_mae_differences        : DATA_NOT_AVAILABLE
  per_observation_winner        : DATA_NOT_AVAILABLE
  mean_paired_error_difference  : DATA_NOT_AVAILABLE
  empirical_80_coverage         : DATA_NOT_AVAILABLE
  empirical_90_coverage         : DATA_NOT_AVAILABLE
  empirical_95_coverage         : DATA_NOT_AVAILABLE
  mean_interval_width           : DATA_NOT_AVAILABLE
  interval_failures             : DATA_NOT_AVAILABLE
  p50_latency_ms                : DATA_NOT_AVAILABLE
  p95_latency_ms                : DATA_NOT_AVAILABLE
  p99

## 9. Conformal Coverage & Interval Verification Readiness
Verify that conformal uncertainty coverage metrics (80%, 90%, 95%) strictly report `DATA_NOT_AVAILABLE` until verified ground truth exists.


In [10]:
print(f"Empirical 80% Coverage: {gov_metrics['empirical_80_coverage']}")
print(f"Empirical 90% Coverage: {gov_metrics['empirical_90_coverage']}")
print(f"Empirical 95% Coverage: {gov_metrics['empirical_95_coverage']}")
print(f"Mean Interval Width:    {gov_metrics['mean_interval_width']}")
print(f"Interval Failures:      {gov_metrics['interval_failures']}")

assert gov_metrics["empirical_80_coverage"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["empirical_90_coverage"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["empirical_95_coverage"] == "DATA_NOT_AVAILABLE"


Empirical 80% Coverage: DATA_NOT_AVAILABLE
Empirical 90% Coverage: DATA_NOT_AVAILABLE
Empirical 95% Coverage: DATA_NOT_AVAILABLE
Mean Interval Width:    DATA_NOT_AVAILABLE
Interval Failures:      DATA_NOT_AVAILABLE


## 10. Population Drift Monitoring Readiness
Verify that drift metrics (PSI, KS, TVD) report `DATA_NOT_AVAILABLE` until live production traffic accumulates.


In [11]:
print(f"PSI:          {gov_metrics['psi']}")
print(f"KS Statistic: {gov_metrics['ks_statistic']}")
print(f"TVD:          {gov_metrics['tvd']}")
print(f"Drift Status: {gov_metrics['drift_status']}")

assert gov_metrics["psi"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["ks_statistic"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["tvd"] == "DATA_NOT_AVAILABLE"
assert gov_metrics["drift_status"] == "DATA_NOT_AVAILABLE"


PSI:          DATA_NOT_AVAILABLE
KS Statistic: DATA_NOT_AVAILABLE
TVD:          DATA_NOT_AVAILABLE
Drift Status: DATA_NOT_AVAILABLE


## 11. Governance Readiness Summary & Promotion Decision
Generate formal governance readiness audit matrix.


In [12]:
summary_df = pd.DataFrame([
    {"Gate": "1. 14-Day Shadow Duration", "Required": ">= 14 days", "Observed": f"{elapsed_days:.2f} days", "Status": "ACTIVE / BLOCKED"},
    {"Gate": "2. Verified Post-Deployment Labels", "Required": ">= 100 labels", "Observed": f"{verified_count} labels", "Status": "BLOCKED"},
    {"Gate": "3. Paired Production Observations", "Required": ">= 100 pairs", "Observed": f"{paired_count} pairs", "Status": "BLOCKED"},
    {"Gate": "4. Point Prediction Metrics", "Required": "Computable", "Observed": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Gate": "5. Conformal Coverage Metrics", "Required": "Computable", "Observed": "DATA_NOT_AVAILABLE", "Status": "BLOCKED"},
    {"Gate": "6. Operational Latency SLA", "Required": "P95 < 150 ms", "Observed": "77.58 ms (Benchmark)", "Status": "PASS (Benchmark)"},
    {"Gate": "7. Candidate Error Isolation", "Required": "0% crash impact", "Observed": "0 exceptions leaked", "Status": "PASS"},
    {"Gate": "8. Human Governance Sign-Off", "Required": "Explicit Approval", "Observed": "PENDING", "Status": "BLOCKED"},
    {"Gate": "9. Model Registry Policy", "Required": "No Auto-Promotion", "Observed": "DISABLED", "Status": "ENFORCED"},
    {"Gate": "10. Model Artifact Integrity", "Required": "SHA-256 match", "Observed": "100% Invariant", "Status": "PASS"}
])

print(summary_df.to_string(index=False))

print("\n" + "="*70)
print("GOVERNANCE DECISION: OPTION C — INSUFFICIENT EVIDENCE")
print("STATUS: REAL-WORLD EVIDENCE ACCUMULATION IN PROGRESS")
print("PROMOTION ELIGIBILITY: FALSE (STRICTLY BLOCKED)")
print("CHAMPION: ACTIVE IN PRODUCTION (phase5_tuned_extra_trees.joblib)")
print("CANDIDATE: VALIDATING IN SHADOW (candidate_v1_2_revalidated.joblib)")
print("AUTOMATIC RETRAINING: DISABLED")
print("AUTOMATIC PROMOTION: DISABLED")
print("MODEL MODIFICATION: NONE")
print("="*70)


                              Gate          Required             Observed           Status
         1. 14-Day Shadow Duration        >= 14 days            1.07 days ACTIVE / BLOCKED
2. Verified Post-Deployment Labels     >= 100 labels             0 labels          BLOCKED
 3. Paired Production Observations      >= 100 pairs              0 pairs          BLOCKED
       4. Point Prediction Metrics        Computable   DATA_NOT_AVAILABLE          BLOCKED
     5. Conformal Coverage Metrics        Computable   DATA_NOT_AVAILABLE          BLOCKED
        6. Operational Latency SLA      P95 < 150 ms 77.58 ms (Benchmark) PASS (Benchmark)
      7. Candidate Error Isolation   0% crash impact  0 exceptions leaked             PASS
      8. Human Governance Sign-Off Explicit Approval              PENDING          BLOCKED
          9. Model Registry Policy No Auto-Promotion             DISABLED         ENFORCED
      10. Model Artifact Integrity     SHA-256 match       100% Invariant             PASS